In [1]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

geemap.ee_initialize()

In [23]:
date_start = ee.Date('2013-08-05') # Earliest station: ZAC
date_end_mod = ee.Date('2013-08-27') # Before orbital drift TERRA
date_end_myd = ee.Date('2013-08-27') # Before orbital drift AQUA

day_diff = date_end_myd.difference(date_start, 'day').getInfo()
print(f'Days between start date and end: {day_diff}')

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)

landmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0).And(ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(0))

icemask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(1)

greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])


Days between start date and end: 22


In [ ]:
# map1 = geemap.Map()
# map1.addLayer(greenlandmask, {'color': 'red'}, 'greenlandmask')
# map1.addLayer(landmask, {'color': 'green'}, 'landmask')
# map1.addLayer(icemask, {'color': 'blue'}, 'icemask')

# map1

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [3]:
sheet = pd.read_csv('./Data/Availability_pattern_sensors.csv')
patterns = pd.DataFrame(sheet)

coef_ice = pd.read_csv('./Data/SKL_Coefficients_Ice_comb.csv')
coef_land = pd.read_csv('./Data/SKL_Coefficients_Land_comb.csv')

lookup_ice = coef_ice.join(patterns.set_index('Sensor_Combo'), on ='Combination').set_index('ID').sort_index().iloc[:,:10]
lookup_land = coef_land.join(patterns.set_index('Sensor_Combo'), on ='Combination').set_index('ID').sort_index().iloc[:,:10]

# Create a dictionary with ID as key and coef_before, intercept_before and combination as values
lookup_ice_dict = ee.Dictionary(lookup_ice.to_dict(orient='index'))
lookup_land_dict = ee.Dictionary(lookup_land.to_dict(orient='index'))
# lookup_land_dict = lookup_land.to_dict(orient='index')

# # 3.2 Calculate Availability Pattern
# def calculate_availability(aqua_day, aqua_night, terra_day, terra_night, viirs_day, viirs_night, jaxa_a, jaxa_b):
#     return (aqua_day.mask().multiply(1)
#         .add(aqua_night.mask().multiply(2))
#         .add(terra_day.mask().multiply(4))
#         .add(terra_night.mask().multiply(8))
#         .add(viirs_day.mask().multiply(16))
#         .add(viirs_night.mask().multiply(32))
#         .add(jaxa_a.mask().multiply(64))
#         .add(jaxa_b.mask().multiply(128))
#     )

# print(lookup_ice)

lookup_land_dict.get('43')

In [4]:
# Mask data based on quality flags

def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)


# def maskJaxa(image):
    # '''Function to filter JAXA GCOM-C LST data based on quality flag.'''
    # qa = image.select('LST_QA_flag')
    # #0: water (land fraction = 0%)
    # #1: mostly water (0% < land fraction < 50%)
    # #2: mostly coastal (50% < land fraction < 100%) - included
    # #3: land (land fraction = 100%) - included
    # mask = qa.gt(1)
    # return image.updateMask(greenlandmask)

def maskViirs(image):
    '''Function to filter VIIRS LST data based on quality flag.'''
    qa = image.select('QC')
    bits01Mask = bitwiseExtract(qa, 0, 1).eq(0); 
    # Bits 0-1: Mandatory QA flags
    # 0: Pixel produced, good quality, no further QA info necessary
    # 1: Pixel produced but unreliable quality
    # 2: Pixel not produced due to cloud
    # 3: Pixel not produced due to reasons other than cloud

    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 2-3: Data quality flag
    # 0: Good data quality of L1B bands 14, 15, 16
    # 1: Missing pixel
    # 2: Fairly calibrated
    # 3: Poorly calibrated, TES processing skipped

    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bits 4-5: Cloud Flag
    # 0: Cloud-free
    # 1: Thin cirrus
    # 2: Pixel within 2 pixels of nearest cloud
    # 3: Cloudy pixels

    # Bits 6-7: Iterations
    # 0: Slow convergence
    # 1: Nominal
    # 2: Nominal
    # 3: Fast

    # Bits 8-9: Atmospheric Opacity
    # 0: ≥3 (Warm, humid air; or cold land)
    # 1: 0.2 - 0.3 (Nominal value)
    # 2: 0.1 - 0.2 (Nominal value)
    # 3: <0.1 (Dry, or high altitude pixel)

    # Bits 10-11: MMD
    # 0: >0.15 (Most silicate rocks)
    # 1: 0.1 - 0.15 (Rocks, sand, some soils)
    # 2: 0.03 - 0.1 (Mostly soils, mixed pixel)
    # 3: <0.03 (Vegetation, snow, water, ice, some soils)

    bit1213Mask = bitwiseExtract(qa, 12, 13).gte(2)
    # Bits 12-13: Emissivity accuracy
    # 0: >0.02 (Poor performance)
    # 1: 0.015 - 0.02 (Marginal performance)
    # 2: 0.01 - 0.015 (Good performance)
    # 3: <0.01 (Excellent performance)

    bit1415Mask = bitwiseExtract(qa, 14, 15).gte(2)
    # Bits 14-15: LST accuracy
    # 0: >2K (Poor performance)
    # 1: 1.5 - 2K (Marginal performance)
    # 2: 1 - 1.5K (Good performance)
    # 3: <1K (Excellent performance)
    
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit1213Mask).And(bit1415Mask)
    return image.updateMask(mask)



In [5]:
# Functions for band selection and conversion

# ERA5 (full scale)
def era5_t2m(image):
    'ERA5 2m air temperature conversion'
    t2m = image.select('mean_2m_air_temperature').subtract(273.15).rename('ERA5_T2m')
    return image.addBands(t2m)

# MODIS
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    qa_day = image.select('QC_Day').rename('MOD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    qa_night = image.select('QC_Night').rename('MOD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    qa_day = image.select('QC_Day').rename('MYD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    qa_night = image.select('QC_Night').rename('MYD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)


# JAXA
def lst_jaxa_a(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_A')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)

def lst_jaxa_d(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_D')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)


# VIIRS 
def lst_viirs_d(image):
    'VIIRS band selection and conversion'
    lst = image.select('LST_1KM').subtract(273.15).rename('VIIRS_LST_D')
    return image.addBands(lst)

def lst_viirs_n(image):
    'VIIRS band selection and conversion'
    lst = image.select('LST_1KM').subtract(273.15).rename('VIIRS_LST_N')
    return image.addBands(lst)



# Load MODIS Terra and Aqua data, apply quality control and conversion functions

ERA5 = (
    ee.ImageCollection("ECMWF/ERA5/DAILY")
    .select(['mean_2m_air_temperature'])
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    .map(era5_t2m)
)

MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_myd_night)
)

JAXA_A = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'A')) # Filter for ascending (AM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) # *************** NEW OBS! CHECK IF CORRECT
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    # .map(maskJaxa)
    .map(lst_jaxa_a)
)

JAXA_D = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) # Filter for descending (PM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) # *************** NEW OBS! CHECK IF CORRECT
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    # .map(maskJaxa)
    .map(lst_jaxa_d)
)


VIIRS_Day = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs_d)
)


VIIRS_Night = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1N") 
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs_n)
)


# print('MODIS Terra Day images:', MOD11A1Daytime.size().getInfo())
# print('MODIS Terra Night images:', MOD11A1Nighttime.size().getInfo())
# print('MODIS Aqua Day images:', MYD11A1Daytime.size().getInfo())
# print('MODIS Aqua Night images:', MYD11A1Nighttime.size().getInfo())
# print('Images in collection Jaxa A', JAXA_A.size().getInfo())     
# print('Images in collection Jaxa B', JAXA_D.size().getInfo())   
# print('Images in collection VIIRS Day', VIIRS_Day.size().getInfo())  
# print('Images in collection VIIRS Night', VIIRS_Night.size().getInfo())

# MODIS Terra Day images: 6032
# MODIS Terra Night images: 6032
# MODIS Aqua Day images: 6435
# MODIS Aqua Night images: 6435
# Images in collection Jaxa A 1125
# Images in collection Jaxa B 1125
# Images in collection VIIRS Day 4693
# Images in collection VIIRS Night 4694




In [6]:
ERA5 = (
    ee.ImageCollection("ECMWF/ERA5/DAILY")
    .select(['mean_2m_air_temperature'])
    .filterDate(date_start, '2024-12-31')
    .filterBounds(greenland)
    .map(era5_t2m)
)

# print(ERA5.getInfo())

In [7]:
def create_date(offset):
    return date_start.advance(offset,'day')

def format_date(date):
    return ee.Date(date).format('YYYY-MM-dd')

dates_list = (ee.List.sequence(0, day_diff-1, 1)
              .map(create_date)
              .map(format_date)
)

# print('Dates list example:', dates_list.getInfo()[100:105])


In [8]:
# ImageCollection.linkCollection(imageCollection, linkedBands, linkedProperties, matchPropertyName)

# Create a common matching property ("date") on both collections, then link by that key.
def add_date_prop(img):
    return img.set('date', ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'))

ERA5_by_date = ERA5.map(add_date_prop)
MOD11A1Daytime_by_date = MOD11A1Daytime.map(add_date_prop)
MOD11A1Nighttime_by_date = MOD11A1Nighttime.map(add_date_prop)
MYD11A1Daytime_by_date = MYD11A1Daytime.map(add_date_prop)
MYD11A1Nighttime_by_date = MYD11A1Nighttime.map(add_date_prop)
JAXA_A_by_date = JAXA_A.map(add_date_prop)
JAXA_D_by_date = JAXA_D.map(add_date_prop)
VIIRS_Day_by_date = VIIRS_Day.map(add_date_prop)
VIIRS_Night_by_date = VIIRS_Night.map(add_date_prop)

sat_stack = (
    ERA5_by_date.select('ERA5_T2m')
    .linkCollection(
        MOD11A1Daytime_by_date.select('MOD_LST_Day'),
        linkedBands=['MOD_LST_Day'],
        linkedProperties=['system:time_start'],
        matchPropertyName='date'
    )
    .linkCollection(MOD11A1Nighttime_by_date.select('MOD_LST_Night'), linkedBands=['MOD_LST_Night'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(MYD11A1Daytime_by_date.select('MYD_LST_Day'), linkedBands=['MYD_LST_Day'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(MYD11A1Nighttime_by_date.select('MYD_LST_Night'), linkedBands=['MYD_LST_Night'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(JAXA_A_by_date.select('JAXA_LST_A'), linkedBands=['JAXA_LST_A'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(JAXA_D_by_date.select('JAXA_LST_D'), linkedBands=['JAXA_LST_D'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(VIIRS_Day_by_date.select('VIIRS_LST_D'), linkedBands=['VIIRS_LST_D'], linkedProperties=['system:time_start'], matchPropertyName='date')
    .linkCollection(VIIRS_Night_by_date.select('VIIRS_LST_N'), linkedBands=['VIIRS_LST_N'], linkedProperties=['system:time_start'], matchPropertyName='date')
)


In [ ]:
# # Create a test image
# testmap = geemap.Map()

# test_img = sat_stack.first()
# # print(test_img.getInfo())
# testmap.addLayer(test_img, {}, 'test image')
# testmap

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
# # Calculate Availability Pattern
# def calculateAvailability(aqua_day, aqua_night, terra_day, terra_night, viirs_day, viirs_night, jaxa_a, jaxa_b):
#     '''Calculates a unique pattern ID in bit format based on the availability of the sensors'''
#     return (aqua_day.mask().multiply(1)
#         .add(aqua_night.mask().multiply(2))
#         .add(terra_day.mask().multiply(4))
#         .add(terra_night.mask().multiply(8))
#         .add(viirs_day.mask().multiply(16))
#         .add(viirs_night.mask().multiply(32))
#         .add(jaxa_a.mask().multiply(64))
#         .add(jaxa_b.mask().multiply(128))
#     )

# # Test
# # pull_coefficients(lookup_ice_dict, 45)

# def applyCorrection(image):
#     MODLST_DAY_avail = image.select('MOD_LST_Day')
#     MODLST_NIGHT_avail = image.select('MOD_LST_Night')
#     MYDLST_DAY_avail = image.select('MYD_LST_Day')
#     MYDLST_NIGHT_avail = image.select('MYD_LST_Night')
#     VIIRS_DAY_avail = image.select('VIIRS_LST_D')
#     VIIRS_NIGHT_avail = image.select('VIIRS_LST_N')
#     JAXA_A_avail = image.select('JAXA_LST_A')
#     JAXA_D_avail = image.select('JAXA_LST_D')

#     availPattern = calculateAvailability(
#         MODLST_DAY_avail,
#         MODLST_NIGHT_avail,
#         MYDLST_DAY_avail,
#         MYDLST_NIGHT_avail,
#         VIIRS_DAY_avail,
#         VIIRS_NIGHT_avail,
#         JAXA_A_avail,
#         JAXA_D_avail
#     ) # returns image

#     availPatternUnique = availPattern.reduceRegion(
#         reducer = ee.Reducer.frequencyHistogram(),
#         geometry = greenland,
#         scale = 1000,
#         maxPixels = 1e13,
#         crs = 'EPSG:3413'
#     )

#     # Safely extract the histogram dictionary for the availability band,
#     # convert the string keys to numbers and filter out the "0" pattern
#     band_name = ee.String(availPattern.bandNames().get(0))
#     hist_dict = ee.Dictionary(availPatternUnique.get(band_name))
#     availPatternUniqueKeys = ee.List(hist_dict.keys()) \
#         .map(lambda k: ee.Number.parse(k)) \
#         .filter(ee.Filter.neq('item', 0))                      # discard the pattern with all bands masked

#     # # # It works until here. The output of availPatternUniqueKeys is a list of 63 unique availability patterns. However, the conversion number -> string -> number -> 

#     # calculate the average LST
#     LSTimage = image.select(
#     ['MOD_LST_Day', 'MOD_LST_Night', 'MYD_LST_Day', 'MYD_LST_Night', 'VIIRS_LST_D', 'VIIRS_LST_N', 'JAXA_LST_A', 'JAXA_LST_D',]
#     ).reduce(ee.Reducer.mean())

#     # # # Create a function to process each coefficient pattern

#     def applyCoefficient(key, correctedLST):
#         numericKey = ee.Number(key).toInt()
#         coeff = lookup_land_dict.get(numericKey.format())
#         coeffDict = ee.Dictionary(coeff)
#         intercept = ee.Number(coeffDict.get('intercept_before'))
#         slope = ee.Number(coeffDict.get('coef_before'))

#         patternMask = availPattern.eq(numericKey)
#         calibratedLST = LSTimage.multiply(slope).add(intercept) \
#         .updateMask(patternMask) \
#         .unmask(0)

#         return ee.Image(correctedLST).add(calibratedLST)
    
#     correctedLST = availPatternUniqueKeys.iterate(applyCoefficient, ee.Image(0))

#     return image.addBands([
#         # if no MODIS LST is available, use ERA5 Land skin temperature
#         ee.Image(correctedLST).rename('Corrected_LST').updateMask(greenlandmask),
#         availPattern.rename('Available_Pattern').updateMask(greenlandmask)
#     ])



In [24]:
# Calculate Availability Pattern
def calculateAvailability(aqua_day, aqua_night, terra_day, terra_night, viirs_day, viirs_night, jaxa_a, jaxa_b):
    '''Calculates a unique pattern ID in bit format based on the availability of the sensors'''
    return (aqua_day.mask().multiply(1)
        .add(aqua_night.mask().multiply(2))
        .add(terra_day.mask().multiply(4))
        .add(terra_night.mask().multiply(8))
        .add(viirs_day.mask().multiply(16))
        .add(viirs_night.mask().multiply(32))
        .add(jaxa_a.mask().multiply(64))
        .add(jaxa_b.mask().multiply(128))
    )

# Test
# pull_coefficients(lookup_ice_dict, 45)

def applyCorrection(image):
    MODLST_DAY_avail = image.select('MOD_LST_Day')
    MODLST_NIGHT_avail = image.select('MOD_LST_Night')
    MYDLST_DAY_avail = image.select('MYD_LST_Day')
    MYDLST_NIGHT_avail = image.select('MYD_LST_Night')
    VIIRS_DAY_avail = image.select('VIIRS_LST_D')
    VIIRS_NIGHT_avail = image.select('VIIRS_LST_N')
    JAXA_A_avail = image.select('JAXA_LST_A')
    JAXA_D_avail = image.select('JAXA_LST_D')

    availPattern = calculateAvailability(
        MODLST_DAY_avail,
        MODLST_NIGHT_avail,
        MYDLST_DAY_avail,
        MYDLST_NIGHT_avail,
        VIIRS_DAY_avail,
        VIIRS_NIGHT_avail,
        JAXA_A_avail,
        JAXA_D_avail
    ) # returns image

    availPatternUnique = availPattern.reduceRegion(
        reducer = ee.Reducer.frequencyHistogram(),
        geometry = greenland,
        scale = 1000,
        maxPixels = 1e13,
        crs = 'EPSG:3413'
    )

    # Safely extract the histogram dictionary for the availability band,
    # convert the string keys to numbers and filter out the "0" pattern
    band_name = ee.String(availPattern.bandNames().get(0))
    hist_dict = ee.Dictionary(availPatternUnique.get(band_name))
    availPatternUniqueKeys = ee.List(hist_dict.keys()) \
        .map(lambda k: ee.Number.parse(k)) \
        .filter(ee.Filter.neq('item', 0))                      # discard the pattern with all bands masked

    # # # It works until here. The output of availPatternUniqueKeys is a list of 63 unique availability patterns. However, the conversion number -> string -> number -> 

    # calculate the average LST
    LSTimage = image.select(
    ['MOD_LST_Day', 'MOD_LST_Night', 'MYD_LST_Day', 'MYD_LST_Night', 'VIIRS_LST_D', 'VIIRS_LST_N', 'JAXA_LST_A', 'JAXA_LST_D',]
    ).reduce(ee.Reducer.mean())

    # # # Create a function to process each coefficient pattern

    def applyLandCoefficient(key, correctedLST):
        numericKey = ee.Number(key).toInt()
        coeff = lookup_land_dict.get(numericKey.format())
        coeffDict = ee.Dictionary(coeff)
        intercept = ee.Number(coeffDict.get('intercept_before'))
        slope = ee.Number(coeffDict.get('coef_before'))

        patternMask = availPattern.eq(numericKey)
        calibratedLST_land = LSTimage.multiply(slope).add(intercept) \
        .updateMask(patternMask) \
        .unmask(0)

        return ee.Image(correctedLST).add(calibratedLST_land)
    
    def applyIceCoefficient(key, correctedLST):
        numericKey = ee.Number(key).toInt()
        coeff = lookup_ice_dict.get(numericKey.format())
        coeffDict = ee.Dictionary(coeff)
        intercept = ee.Number(coeffDict.get('intercept_before'))
        slope = ee.Number(coeffDict.get('coef_before'))

        patternMask = availPattern.eq(numericKey)
        calibratedLST_ice = LSTimage.multiply(slope).add(intercept) \
        .updateMask(patternMask) \
        .unmask(0)

        return ee.Image(correctedLST).add(calibratedLST_ice)

    correctedLST_land = availPatternUniqueKeys.iterate(applyLandCoefficient, ee.Image(0))
    correctedLST_ice = availPatternUniqueKeys.iterate(applyIceCoefficient, ee.Image(0))

    correctedLST = ee.Image(correctedLST_land).updateMask(landmask).unmask(0).add(
        ee.Image(correctedLST_ice).updateMask(icemask).unmask(0)
    )

    return image.addBands([
        ee.Image(correctedLST).rename('Corrected_LST').updateMask(greenlandmask),
        availPattern.rename('Available_Pattern').updateMask(greenlandmask)
    ])



In [25]:
test_img = sat_stack.first()

corrected = applyCorrection(test_img)

# test_img_corr_applied = applyCorrection(test_img)

# # test_img_corr_applied

visparams = {
    'min': -30,
    'max': 30,
    'palette': 'coolwarm',
    'bands': ['Corrected_LST']
}

map2 = geemap.Map()

map2.addLayer(corrected, visparams, 'corrected sat_stack first image')
map2.centerObject(greenland, 3)

map2

Map(center=[72.70302525720832, -41.7868898523626], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
# download corrected image

task = ee.batch.Export.image.toDrive(
    image=corrected.select('Corrected_LST'),
    description='Corrected_LST',
    folder='GEMLST_MODIS',
    fileNamePrefix='Corrected_LST_First_Image',
    scale=1000,
    region=greenland,
    crs='EPSG:3413',
)
